# Session 2 — I-VT, AOI/TOI metrics, heatmaps & gaze plots
**90 minutes**

### Goals
1. Convert timestamps correctly, then run **I-VT**.
2. Build fixation events with **rational durations**.
3. Compute **AOI dwell** and **TTFF relative to stimulus onset** (not absolute clock time).
4. Show **count** vs **duration** heatmaps and a Tobii-like gaze plot.

Only one helper function is imported: `ivt_classify`. Everything else is written inline so you can see the math.


In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Readable plots for projection
plt.rcParams.update({
    "figure.figsize": (7, 4),
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
})

SESSION_DIR = Path.cwd()
WORKSHOP_DIR = SESSION_DIR.parent if SESSION_DIR.name == "sessions" else Path("workshop")
sys.path.insert(0, str(WORKSHOP_DIR))
from analysis.paths import data_path, stimuli_path
from analysis.ivt import ivt_classify
from PIL import Image
from scipy.ndimage import gaussian_filter


## 1. Load gaze for one stimulus (cake)

**Timestamp rule for this export:** `Recording timestamp` is in **milliseconds**.  
We divide by `1e3` to get seconds. Always check median Δt (should be ~0.008–0.020 s for ~60–120 Hz).


In [ ]:
raw = pd.read_csv(data_path("food_decision_making", "Food_Decision_Making_Teaching_Sample.csv"))
gaze = raw.loc[raw["Sensor"] == "Eye Tracker"].copy()
gaze = gaze.dropna(subset=["Gaze point X", "Gaze point Y", "Recording timestamp"])
gaze["time_s"] = gaze["Recording timestamp"].astype(float) / 1e3
gaze = gaze.sort_values("time_s")

stim = "cake"
g = gaze.loc[gaze["Presented Stimulus name"] == stim].copy()
toi_onset = g["time_s"].iloc[0]
toi_offset = g["time_s"].iloc[-1]

qc = pd.DataFrame({
    "stimulus": [stim],
    "n_samples": [len(g)],
    "median_dt_s": [g["time_s"].diff().median()],
    "toi_duration_s": [toi_offset - toi_onset],
})
qc

## 2. I-VT classification

Velocity at sample *i* ≈ distance(i-1 → i) / Δt.  
Fixation if velocity < threshold; then remove fixation runs shorter than 60 ms.


In [ ]:
samples, events = ivt_classify(
    g["time_s"].to_numpy(),
    g["Gaze point X"].to_numpy(),
    g["Gaze point Y"].to_numpy(),
    velocity_threshold=5000,  # px/s for this screen; try 2000 and 8000 later
    min_fixation_s=0.06,
)

label_counts = samples["label"].value_counts().rename_axis("label").reset_index(name="n_samples")
label_counts

In [ ]:
fix = events.loc[events["label"] == "fixation"].copy()
sac = events.loc[events["label"] == "saccade"].copy()

# Sanity: fixation dwell cannot exceed the stimulus window by much
event_summary = pd.DataFrame([
    {"metric": "n_fixations", "value": len(fix)},
    {"metric": "n_saccades", "value": len(sac)},
    {"metric": "mean_fix_duration_s", "value": round(fix["duration_s"].mean(), 3) if len(fix) else np.nan},
    {"metric": "total_fixation_dwell_s", "value": round(fix["duration_s"].sum(), 3) if len(fix) else 0.0},
    {"metric": "dwell_fraction_of_TOI", "value": round(fix["duration_s"].sum() / (toi_offset - toi_onset), 3) if len(fix) else 0.0},
])
event_summary

In [ ]:
fig, ax = plt.subplots()
ax.bar(["fixations", "saccades"], [len(fix), len(sac)], color=["#2a6f6f", "#b85c38"])
ax.set_ylabel("Number of events")
ax.set_title(f"I-VT events on '{stim}'")
plt.tight_layout()
plt.show()

### Compare with Tobii labels (sample level, compact)
We compare **sample labels**, not event counts (different segmentation → counts need not match).


In [ ]:
cmp = pd.DataFrame({
    "tobii": g["Eye movement type"].to_numpy(),
    "ivt": samples["label"].to_numpy(),
})
# Agreement only on rows where Tobii says Fixation or Saccade
both = cmp[cmp["tobii"].isin(["Fixation", "Saccade"])].copy()
both["agree"] = (
    ((both["tobii"] == "Fixation") & (both["ivt"] == "fixation"))
    | ((both["tobii"] == "Saccade") & (both["ivt"] == "saccade"))
)
agree_tbl = pd.DataFrame({
    "n_compared_samples": [len(both)],
    "pct_agree": [round(100 * both["agree"].mean(), 1)],
})
agree_tbl

## 3. AOI assignment + metrics (inline)

Teaching AOIs are approximate rectangles on 1366×768 images.  
**TTFF** = time of first fixation in that AOI **minus stimulus/TOI onset** (not absolute recording time).


In [ ]:
# Pixel boxes: (x_min, y_min, x_max, y_max), origin top-left
aois = {
    "food-pic": (430, 140, 930, 520),
    "buy": (180, 560, 520, 700),
    "not-buy": (840, 560, 1180, 700),
}

def which_aoi(x, y):
    for name, (x0, y0, x1, y1) in aois.items():
        if (x0 <= x <= x1) and (y0 <= y <= y1):
            return name
    return pd.NA

fix["aoi"] = [which_aoi(x, y) for x, y in zip(fix["x_mean"], fix["y_mean"])]
fix["ttff_from_onset_s"] = fix["start_s"] - toi_onset
fix[["start_s", "duration_s", "x_mean", "y_mean", "aoi", "ttff_from_onset_s"]].head(8)

In [ ]:
# Per-AOI metrics (only AOIs that were hit)
rows = []
for aoi_name, grp in fix.dropna(subset=["aoi"]).groupby("aoi"):
    rows.append({
        "AOI": aoi_name,
        "n_fixations": len(grp),
        "dwell_s": round(grp["duration_s"].sum(), 3),
        "mean_fix_s": round(grp["duration_s"].mean(), 3),
        "TTFF_s": round(grp["ttff_from_onset_s"].min(), 3),  # earliest fixation in AOI after onset
    })
aoi_metrics = pd.DataFrame(rows).sort_values("dwell_s", ascending=False)
aoi_metrics

In [ ]:
fig, ax = plt.subplots()
ax.bar(aoi_metrics["AOI"], aoi_metrics["dwell_s"], color="#345995")
ax.set_ylabel("Dwell (s)")
ax.set_title(f"AOI dwell during '{stim}' (I-VT fixations)")
plt.tight_layout()
plt.show()

## 4. Heatmaps (count vs duration) — one stimulus, two panels

In [ ]:
img = np.asarray(Image.open(stimuli_path("Decision Making", f"{stim}.png")).convert("RGBA"))
h, w = img.shape[0], img.shape[1]
xs = fix["x_mean"].to_numpy()
ys = fix["y_mean"].to_numpy()
durs = fix["duration_s"].to_numpy()

def make_heat(weights=None):
    heat, _, _ = np.histogram2d(xs, ys, bins=50, range=[[0, w], [0, h]], weights=weights)
    heat = gaussian_filter(heat.T.astype(float), sigma=1.5)
    return heat

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
for ax, weights, title in [
    (axes[0], None, "Fixation COUNT heatmap"),
    (axes[1], durs, "Fixation DURATION heatmap"),
]:
    ax.imshow(img)
    ax.imshow(make_heat(weights), extent=[0, w, h, 0], alpha=0.45, cmap="jet", interpolation="bilinear")
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 5. Tobii-like gaze plot (order + duration)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.imshow(img)
fx = fix.sort_values("start_s")
ax.plot(fx["x_mean"], fx["y_mean"], color="black", linewidth=1, alpha=0.7)
# Circle radius proportional to duration (readable, not huge)
max_r = 36
scale = max_r / max(fx["duration_s"].max(), 1e-6)
for i, (_, r) in enumerate(fx.iterrows(), start=1):
    rad = max(6, r["duration_s"] * scale)
    ax.add_patch(plt.Circle((r["x_mean"], r["y_mean"]), rad, facecolor="#e41a1c88", edgecolor="#7f0000"))
    ax.text(r["x_mean"], r["y_mean"], str(i), ha="center", va="center", color="white", fontsize=7)
ax.set_title(f"Gaze plot — {stim} (numbers = fixation order)")
ax.axis("off")
plt.tight_layout()
plt.show()

# Keep the plot readable: if many fixations, show only first 12 in a table
fx[["duration_s", "aoi"]].head(12)

## Practice
1. Re-run I-VT with threshold 2000 and 8000. Watch `n_fixations` and `dwell_fraction_of_TOI`.
2. Why is TTFF from onset more interpretable than absolute `start_s`?
3. Exit ticket: paste your `aoi_metrics` table for cake.
